## Imports

Resources used: 
- https://www.livercellatlas.org/umap-cd45pos.php
- https://www.livercellatlas.org/umap-ststmouseAll.php


In [ ]:
import logging, warnings; logging.getLogger().setLevel(logging.ERROR);
warnings.filterwarnings("ignore")

import scanpy as sc
import scanpy.external as sce
import numpy as np
import pandas as pd
import re
from pathlib import Path 

import warnings, scipy.sparse as sp, matplotlib, matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.pyplot import rc_context
import matplotlib.font_manager
import matplotlib.lines as lines


pd.set_option('display.max_rows', 200)

matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = 'Arial'
matplotlib.rc('font', size=12)

sc.settings.n_jobs=-1
sc.set_figure_params(dpi=80, dpi_save=300, color_map='Spectral_r', vector_friendly=True, transparent=True)
sc.settings.figdir = '../../1_outputs/0_figures'
sc.settings.verbosity = 1 # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()

%matplotlib inline 
%config InlineBackend.figure_format = 'retina'

In [ ]:
# preset color palettes and color maps
user_defined_palette =  [ '#F6222E', '#16FF32', '#3283FE', '#FEAF16', '#BDCDFF', '#3B00FB', '#1CFFCE', '#C075A6', '#F8A19F', '#B5EFB5', '#FBE426', '#C4451C', 
                          '#2ED9FF', '#c1c119', '#8b0000', '#FE00FA', '#1CBE4F', '#1C8356', '#0e452b', '#AA0DFE', '#B5EFB5', '#325A9B', '#90AD1C']

user_defined_cmap_markers = LinearSegmentedColormap.from_list('mycmap', ["#E6E6FF", "#CCCCFF", "#B2B2FF", "#9999FF",  "#6666FF",   "#3333FF", "#0000FF"])
user_defined_cmap_degs = LinearSegmentedColormap.from_list('mycmap', ["#0000FF", "#3333FF", "#6666FF", "#9999FF", "#B2B2FF", "#CCCCFF", "#E6E6FF", "#E6FFE6", "#CCFFCC", "#B2FFB2", "#99FF99", "#66FF66", "#33FF33", "#00FF00"])

In [ ]:
pwd

In [ ]:
file_outputs = '../../../1_outputs/' 
h5ad = '../../../3_h5ad/'

In [ ]:
## read in h5ad 

adata = sc.read_h5ad(h5ad + "3_liver.h5ad")
adata

## Rerun HVG, PCA, Harmony, UMAP

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=3000, n_bins=20, flavor='seurat',  inplace=True)

In [ ]:
sc.tl.pca(adata, n_comps=45, svd_solver='arpack', random_state=42, use_highly_variable=True) #change use_highly_variable == True if you ran the block above

#### Harmony integrate the data if there are large batch effects

In [ ]:
#sce.pp.harmony_integrate(adata, 'condition') #Usually you want to run based on replicates but you can also run based on other parameters

If you batch corrected, make sure to use the use_rep = 'X_pca_harmony' paremeter below

In [ ]:
rng = np.random.RandomState(42)
sc.pp.neighbors(adata, n_neighbors=30, random_state=42, use_rep='X_pca')  # Change use_rep == X_pca_harmony if you ran the block above

In [ ]:
sc.tl.umap(adata, random_state=42)#, min_dist = 0.3, spread = 1.1)

In [ ]:
sc.set_figure_params(dpi=80, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    adata, 
    color=['sample', 'tissue', 'condition', 'Cd4'],   
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.5,
    outline_width=[0.6, 0.05],
    size=5,
    frameon=False,
    add_outline=True,
    sort_order = False
)

# Annotations

In [ ]:
sc.set_figure_params(dpi=80, dpi_save=300, color_map='viridis', vector_friendly=True, transparent=True)
sc.pl.umap(
    adata, 
    color=[
        'sample', 

        'Ptprc',

        #Early Lymphoid Progentors
        'Kit', 'Ly6a', 'Il7r', 'Rag1', 'Bcl11a', 'Cd44',

        #T cells: 
        'Cd3e', 'Cd4', 'Cd8a', 'Cd8b1', 'Foxp3', 'Nkg7', 'Gzmb', 'Gzmk', 'Il7r',

        #Early T
        'Lef1', 'Sell', 'Tcf7',

        #Cd4 Th1
        'Tbx21', 'Ifng',

        #Cd4 Th2 
        'Gata3', 'Ifng',

        #Cd4 Th17
        'Rorc', 'Il17a',

        #Cd4 Th22
        'Ahr', 'Il22',

        #NK cells
        'Klrb1c',
        
        #B Cells:
        'Cd19', 'Ighd', 'Cr2',

        #Plasma Cells: 
        'Tnfrsf17', 'Sdc1', 

        #Myeloid 
        'Itgam', 'Itgax',

        #Proliferating
        'Mki67',

        #DCs
        'Itgax', 'Sirpa', 'Siglech', 'Flt3', 'Csf1r', 'Clec9a',

        #Kuppfer Cells 
        'Adgre1', 'Clec4f'

        ],  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=4,
    wspace = 0.3,
    outline_width=[0.6, 0.05],
    vmax='p99',
    frameon=False,
    add_outline=True,
    sort_order = False,
)

In [ ]:
#0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 
#1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0

for resolution_parameter in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    sc.tl.leiden(adata, resolution=resolution_parameter, random_state=42, key_added='leiden_'+str(resolution_parameter))

In [ ]:
#'leiden_0.1', 'leiden_0.2', 'leiden_0.3','leiden_0.4', 'leiden_0.5', 'leiden_0.6', 'leiden_0.8', 'leiden_0.9', 'leiden_1.0',
#'leiden_1.1', 'leiden_1.2', 'leiden_1.3','leiden_1.4', 'leiden_1.5', 'leiden_1.6', 'leiden_1.7', 'leiden_1.8', 'leiden_1.9', 'leiden_2.0'

sc.set_figure_params(dpi=80, dpi_save=300, color_map='viridis', vector_friendly=True, transparent=True)
sc.pl.umap(
    adata, 
    color=[
        'leiden_0.1', 'leiden_0.2', 'leiden_0.3', 'leiden_0.4', 'leiden_0.5',
        'leiden_0.6', 'leiden_0.7', 'leiden_0.8', 'leiden_0.9', 'leiden_1.0',
           ], 
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=4,
    wspace = 0.7,
    outline_width=[0.6, 0.05],
    #size=35,
    frameon=False,
    add_outline=True,
    sort_order = False
)

In [ ]:
sc.tl.rank_genes_groups(adata, 
                        groupby='leiden_0.1',  # Change the Leiden clustering
                        method='wilcoxon', 
                        use_raw=False)

result = adata.uns['rank_genes_groups']
groups = result['names'].dtype.names

df = pd.DataFrame(
    {group + '_' + key[:1]: result[key][group]
    for group in groups for key in ['names']}).head(150)

#df.to_csv(file_outputs + '1_liver_leiden0.6.csv'', index = False)

df.head(25)

In [ ]:
#'leiden_0.1', 'leiden_0.2', 'leiden_0.3','leiden_0.4', 'leiden_0.5', 'leiden_0.6', 'leiden_0.8', 'leiden_0.9', 'leiden_1.0',
#'leiden_1.1', 'leiden_1.2', 'leiden_1.3','leiden_1.4', 'leiden_1.5', 'leiden_1.6', 'leiden_1.7', 'leiden_1.8', 'leiden_1.9', 'leiden_2.0'

sc.set_figure_params(dpi=80, dpi_save=300, color_map='viridis', vector_friendly=True, transparent=True)
sc.pl.umap(
    adata, 
    color=[
        'leiden_0.1', 'Cd4', 'Clnk', 'Tmem176a'
           ], 
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=4,
    wspace = 0.7,
    outline_width=[0.6, 0.05],
    #size=35,
    frameon=False,
    add_outline=True,
    sort_order = False
)

In [ ]:
cell_type_groups = {
    'Subset1' : ['0', '3'], 
    'Subset2' : ['1', '2'],
    'Subset3' : ['4', '5', '6']
    # add more as needed
}

cluster_to_cell_type = {cluster: cell_type for cell_type, clusters in cell_type_groups.items() for cluster in clusters}

In [ ]:
adata.obs['cell_type'] = adata.obs['leiden_0.1'].map(cluster_to_cell_type) #Add Leiden Cluster

In [ ]:
sc.set_figure_params(dpi=80, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    adata, 
    color=[
        'cell_type', 
          ],
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.3,
    outline_width=[0.6, 0.05],
    frameon=False,
    add_outline=True,
    sort_order = False, 
)

In [ ]:
#adata.write_h5ad(h5ad + '3_liver.h5ad')

### Subset 1

In [ ]:
sub1 = adata[adata.obs['cell_type'].isin(['Subset1'])].copy()
sub1

In [ ]:
sc.pp.highly_variable_genes(sub1, n_top_genes=3000, n_bins=20, flavor='seurat',  inplace=True)

In [ ]:
sc.tl.pca(sub1, n_comps=55, svd_solver='arpack', random_state=42, use_highly_variable=True) #change use_highly_variable == True if you ran the block above

#### Harmony integrate the data if there are large batch effects

In [ ]:
#sce.pp.harmony_integrate(adata, 'condition') #Usually you want to run based on replicates but you can also run based on other parameters

If you batch corrected, make sure to use the use_rep = 'X_pca_harmony' paremeter below

In [ ]:
rng = np.random.RandomState(42)
sc.pp.neighbors(sub1, n_neighbors=30, random_state=42, use_rep='X_pca')  # Change use_rep == X_pca_harmony if you ran the block above

In [ ]:
sc.tl.umap(sub1, random_state=42)#, min_dist = 0.3, spread = 1.1)

In [ ]:
sc.set_figure_params(dpi=80, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    sub1, 
    color=['sample', 'tissue', 'condition', 'Cd4'],   
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.5,
    outline_width=[0.6, 0.05],
    size=5,
    frameon=False,
    add_outline=True,
    sort_order = False
)

In [ ]:
sc.set_figure_params(dpi=80, dpi_save=300, color_map='viridis', vector_friendly=True, transparent=True)
sc.pl.umap(
    sub1, 
    color=[
        'sample', 

        'Ptprc',

        #Early Lymphoid Progentors
        'Kit', 'Ly6a', 'Il7r', 'Rag1', 'Bcl11a', 'Cd44',

        #T cells: 
        'Cd3e', 'Cd4', 'Cd8a', 'Cd8b1', 'Foxp3', 'Nkg7', 'Gzmb', 'Gzmk', 

        #EM 
        'Il7r', 'Cd44', 'Sell', #Negative

        #Progenitor Exhuasted
        'Tcf7', 'Pdcd1', 'Havcr2',

        #Early T
        'Lef1', 'Sell', 'Tcf7',

        #Cd4 Th1
        'Tbx21', 'Ifng',

        #Cd4 Th2 
        'Gata3', 'Ifng',

        #Cd4 Th17
        'Rorc', 'Il17a',

        #Cd4 Th22
        'Ahr', 'Il22',

        ],  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=4,
    wspace = 0.3,
    outline_width=[0.6, 0.05],
    vmax='p99',
    frameon=False,
    add_outline=True,
    sort_order = False,
)

In [ ]:
#0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 
#1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0

for resolution_parameter in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5]:
    sc.tl.leiden(sub1, resolution=resolution_parameter, random_state=42, key_added='leiden_'+str(resolution_parameter))

In [ ]:
#'leiden_0.1', 'leiden_0.2', 'leiden_0.3','leiden_0.4', 'leiden_0.5', 'leiden_0.6', 'leiden_0.8', 'leiden_0.9', 'leiden_1.0',
#'leiden_1.1', 'leiden_1.2', 'leiden_1.3','leiden_1.4', 'leiden_1.5', 'leiden_1.6', 'leiden_1.7', 'leiden_1.8', 'leiden_1.9', 'leiden_2.0'

sc.set_figure_params(dpi=80, dpi_save=300, color_map='viridis', vector_friendly=True, transparent=True)
sc.pl.umap(
    sub1, 
    color=[
        'leiden_0.1', 'leiden_0.2', 'leiden_0.3', 'leiden_0.4', 'leiden_0.5',
        'leiden_0.6', 'leiden_0.7', 'leiden_0.8', 'leiden_0.9', 'leiden_1.0',
        'leiden_1.1', 'leiden_1.2', 'leiden_1.3','leiden_1.4', 'leiden_1.5'
           ], 
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=4,
    wspace = 0.7,
    outline_width=[0.6, 0.05],
    #size=35,
    frameon=False,
    add_outline=True,
    sort_order = False
)

In [ ]:
sc.tl.rank_genes_groups(sub1, 
                        groupby='leiden_1.0',  # Change the Leiden clustering
                        method='wilcoxon', 
                        use_raw=False)

result = sub1.uns['rank_genes_groups']
groups = result['names'].dtype.names

df = pd.DataFrame(
    {group + '_' + key[:1]: result[key][group]
    for group in groups for key in ['names']}).head(150)

#df.to_csv(file_outputs + '1_liver_leiden0.6.csv'', index = False)

df.head(25)

In [ ]:
#'leiden_0.1', 'leiden_0.2', 'leiden_0.3','leiden_0.4', 'leiden_0.5', 'leiden_0.6', 'leiden_0.8', 'leiden_0.9', 'leiden_1.0',
#'leiden_1.1', 'leiden_1.2', 'leiden_1.3','leiden_1.4', 'leiden_1.5', 'leiden_1.6', 'leiden_1.7', 'leiden_1.8', 'leiden_1.9', 'leiden_2.0'

sc.set_figure_params(dpi=80, dpi_save=300, color_map='viridis', vector_friendly=True, transparent=True)
sc.pl.umap(
    sub1, 
    color=[
        'leiden_1.0', 'Cd4', 'Cd8a', 'Sell', 'Lef1', 'Mki67', 'Foxp3',
           ], 
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=2,
    wspace = 0.7,
    outline_width=[0.6, 0.05],
    #size=35,
    frameon=False,
    add_outline=True,
    sort_order = False
)

In [ ]:
cell_type_groups = {
    'T CD4:EM' : ['0', '2', '4'], 
    'T CD8' : ['1', '6'],
    'T Ki67': ['3', '5', '7', '8', '9'], 
    'CD4/CD8': ['10', '11']
    # add more as needed
}

cluster_to_cell_type = {cluster: cell_type for cell_type, clusters in cell_type_groups.items() for cluster in clusters}

In [ ]:
sub1.obs['cell_type'] = sub1.obs['leiden_1.0'].map(cluster_to_cell_type) #Add Leiden Cluster

In [ ]:
sc.set_figure_params(dpi=80, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    sub1, 
    color=[
        'cell_type', 
          ],
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.3,
    outline_width=[0.6, 0.05],
    frameon=False,
    add_outline=True,
    sort_order = False, 
)

#### Annotation from Subset 1: CD4/CD8

In [ ]:
combo = sub1[sub1.obs['cell_type'].isin(['CD4/CD8'])].copy()
combo

In [ ]:
#sc.pp.highly_variable_genes(combo, n_top_genes=3000, n_bins=20, flavor='seurat',  inplace=True)

In [ ]:
sc.tl.pca(combo, n_comps=10, svd_solver='arpack', random_state=42, use_highly_variable=True) #change use_highly_variable == True if you ran the block above

#### Harmony integrate the data if there are large batch effects

In [ ]:
#sce.pp.harmony_integrate(adata, 'condition') #Usually you want to run based on replicates but you can also run based on other parameters

If you batch corrected, make sure to use the use_rep = 'X_pca_harmony' paremeter below

In [ ]:
rng = np.random.RandomState(42)
sc.pp.neighbors(combo, n_neighbors=30, random_state=42, use_rep='X_pca')  # Change use_rep == X_pca_harmony if you ran the block above

In [ ]:
sc.tl.umap(combo, random_state=42)#, min_dist = 0.3, spread = 1.1)

In [ ]:
sc.set_figure_params(dpi=80, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    combo, 
    color=['sample', 'tissue', 'condition', 'Cd4', 'Cd8a', 'Mki67', 'Foxp3',
           'Il18r1'],   
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.5,
    outline_width=[0.6, 0.05],
    #size=5,
    frameon=False,
    add_outline=True,
    sort_order = False
)

In [ ]:
#0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 
#1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0

for resolution_parameter in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5]:
    sc.tl.leiden(combo, resolution=resolution_parameter, random_state=42, key_added='leiden_'+str(resolution_parameter))

In [ ]:
#'leiden_0.1', 'leiden_0.2', 'leiden_0.3','leiden_0.4', 'leiden_0.5', 'leiden_0.6', 'leiden_0.8', 'leiden_0.9', 'leiden_1.0',
#'leiden_1.1', 'leiden_1.2', 'leiden_1.3','leiden_1.4', 'leiden_1.5', 'leiden_1.6', 'leiden_1.7', 'leiden_1.8', 'leiden_1.9', 'leiden_2.0'

sc.set_figure_params(dpi=80, dpi_save=300, color_map='viridis', vector_friendly=True, transparent=True)
sc.pl.umap(
    combo, 
    color=[
        'leiden_0.1', 'leiden_0.2', 'leiden_0.3', 'leiden_0.4', 'leiden_0.5',
        'leiden_0.6', 'leiden_0.7', 'leiden_0.8', 'leiden_0.9', 'leiden_1.0',
           ], 
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=4,
    wspace = 0.7,
    outline_width=[0.6, 0.05],
    #size=35,
    frameon=False,
    add_outline=True,
    sort_order = False
)

In [ ]:
sc.tl.rank_genes_groups(combo, 
                        groupby='leiden_0.9',  # Change the Leiden clustering
                        method='wilcoxon', 
                        use_raw=False)

result = combo.uns['rank_genes_groups']
groups = result['names'].dtype.names

df = pd.DataFrame(
    {group + '_' + key[:1]: result[key][group]
    for group in groups for key in ['names']}).head(150)

df.to_csv(file_outputs + '1_liver_combo_leiden0.9.csv', index = False)

df.head(25)

In [ ]:
#'leiden_0.1', 'leiden_0.2', 'leiden_0.3','leiden_0.4', 'leiden_0.5', 'leiden_0.6', 'leiden_0.8', 'leiden_0.9', 'leiden_1.0',
#'leiden_1.1', 'leiden_1.2', 'leiden_1.3','leiden_1.4', 'leiden_1.5', 'leiden_1.6', 'leiden_1.7', 'leiden_1.8', 'leiden_1.9', 'leiden_2.0'

sc.set_figure_params(dpi=80, dpi_save=300, color_map='viridis', vector_friendly=True, transparent=True)
sc.pl.umap(
    combo, 
    color=[
        'leiden_0.4', 'Cd4', 'Cd8a', 'Il18r1', 'Sell', 'Foxp3', 'Il2ra', 'Ikzf1',
        ], 
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=2,
    wspace = 0.7,
    outline_width=[0.6, 0.05],
    #size=35,
    frameon=False,
    add_outline=True,
    sort_order = False
)

In [ ]:
cell_type_groups = {
    'T Treg' : ['1'],
    'T CD4:EM': ['0'],
    'T CD8:Foxp3+': ['2']
    # add more as needed
}

cluster_to_cell_type = {cluster: cell_type for cell_type, clusters in cell_type_groups.items() for cluster in clusters}

In [ ]:
combo.obs['cell_type'] = ''

In [ ]:
combo.obs['cell_type'] = combo.obs['leiden_0.4'].map(cluster_to_cell_type) #Add Leiden Cluster

In [ ]:
sc.set_figure_params(dpi=80, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    combo, 
    color=[
        'cell_type', 
          ],
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.3,
    outline_width=[0.6, 0.05],
    frameon=False,
    add_outline=True,
    sort_order = False, 
)

### Annotation from Subset1: Mki67+

In [ ]:
sub1_prol = sub1[sub1.obs['cell_type'].isin(['T Ki67'])].copy()
sub1_prol

In [ ]:
#sc.pp.highly_variable_genes(sub1_prol, n_top_genes=3000, n_bins=20, flavor='seurat',  inplace=True)

In [ ]:
sc.tl.pca(sub1_prol, n_comps=45, svd_solver='arpack', random_state=42, use_highly_variable=False) #change use_highly_variable == True if you ran the block above

#### Harmony integrate the data if there are large batch effects

In [ ]:
#sce.pp.harmony_integrate(adata, 'condition') #Usually you want to run based on replicates but you can also run based on other parameters

If you batch corrected, make sure to use the use_rep = 'X_pca_harmony' paremeter below

In [ ]:
rng = np.random.RandomState(42)
sc.pp.neighbors(sub1_prol, n_neighbors=30, random_state=42, use_rep='X_pca')  # Change use_rep == X_pca_harmony if you ran the block above

In [ ]:
sc.tl.umap(sub1_prol, random_state=42)#, min_dist = 0.3, spread = 1.1)

In [ ]:
sc.set_figure_params(dpi=80, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    sub1_prol, 
    color=['sample', 'tissue', 'condition', 'Cd4', 'Cd8a', 'Mki67',
           'Il18r1'],   
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.5,
    outline_width=[0.6, 0.05],
    size=10,
    frameon=False,
    add_outline=True,
    sort_order = False
)

In [ ]:
#0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 
#1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0

for resolution_parameter in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5]:
    sc.tl.leiden(sub1_prol, resolution=resolution_parameter, random_state=42, key_added='leiden_'+str(resolution_parameter))

In [ ]:
#'leiden_0.1', 'leiden_0.2', 'leiden_0.3','leiden_0.4', 'leiden_0.5', 'leiden_0.6', 'leiden_0.8', 'leiden_0.9', 'leiden_1.0',
#'leiden_1.1', 'leiden_1.2', 'leiden_1.3','leiden_1.4', 'leiden_1.5', 'leiden_1.6', 'leiden_1.7', 'leiden_1.8', 'leiden_1.9', 'leiden_2.0'

sc.set_figure_params(dpi=80, dpi_save=300, color_map='viridis', vector_friendly=True, transparent=True)
sc.pl.umap(
    sub1_prol, 
    color=[
        'leiden_0.1', 'leiden_0.2', 'leiden_0.3', 'leiden_0.4', 'leiden_0.5',
        'leiden_0.6', 'leiden_0.7', 'leiden_0.8', 'leiden_0.9', 'leiden_1.0',
        'leiden_1.1', 'leiden_1.2', 'leiden_1.3','leiden_1.4', 'leiden_1.5',
           ], 
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=4,
    wspace = 0.7,
    outline_width=[0.6, 0.05],
    #size=35,
    frameon=False,
    add_outline=True,
    sort_order = False
)

In [ ]:
sc.tl.rank_genes_groups(sub1_prol, 
                        groupby='leiden_0.9',  # Change the Leiden clustering
                        method='wilcoxon', 
                        use_raw=False)

result = sub1_prol.uns['rank_genes_groups']
groups = result['names'].dtype.names

df = pd.DataFrame(
    {group + '_' + key[:1]: result[key][group]
    for group in groups for key in ['names']}).head(150)

!df.to_csv(file_outputs + '1_liver_combo_leiden0.9.csv', index = False)

df.head(25)

In [ ]:
#'leiden_0.1', 'leiden_0.2', 'leiden_0.3','leiden_0.4', 'leiden_0.5', 'leiden_0.6', 'leiden_0.8', 'leiden_0.9', 'leiden_1.0',
#'leiden_1.1', 'leiden_1.2', 'leiden_1.3','leiden_1.4', 'leiden_1.5', 'leiden_1.6', 'leiden_1.7', 'leiden_1.8', 'leiden_1.9', 'leiden_2.0'

sc.set_figure_params(dpi=80, dpi_save=300, color_map='viridis', vector_friendly=True, transparent=True)
sc.pl.umap(
    sub1_prol, 
    color=[
        'leiden_1.4', 'Cd4', 'Cd8a',
        ], 
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=3,
    wspace = 0.7,
    outline_width=[0.6, 0.05],
    #size=35,
    frameon=False,
    add_outline=True,
    sort_order = False
)

In [ ]:
cell_type_groups = {
    'T CD8:Ki67+' : ['0', '1', '3', '5', '6', '8', '9', '11', '12'], 
    'T CD4:Ki67+' : ['4', '2', '7', '10'],
    # add more as needed
}

cluster_to_cell_type = {cluster: cell_type for cell_type, clusters in cell_type_groups.items() for cluster in clusters}

In [ ]:
sub1_prol.obs['cell_type'] = sub1_prol.obs['leiden_1.4'].map(cluster_to_cell_type) #Add Leiden Cluster

In [ ]:
sc.set_figure_params(dpi=80, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    sub1_prol, 
    color=[
        'cell_type', 
          ],
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.3,
    outline_width=[0.6, 0.05],
    frameon=False,
    add_outline=True,
    sort_order = False, 
)

#### Map CD4/CD8 and Ki67+ Cells back to Subset 1

In [ ]:
sub1.obs['cell_type'].unique()

In [ ]:
temp = sub1[~sub1.obs['cell_type'].isin(['CD4/CD8', 'T Ki67'])].obs['cell_type']
temp

In [ ]:
with_renamed_subsets = pd.concat([combo.obs['cell_type'],
                                  sub1_prol.obs['cell_type'],
                                  temp])

In [ ]:
sub1.obs['cell_type_subset'] = ''

In [ ]:
sub1.obs['cell_type_subset'][sub1.obs.index.isin(with_renamed_subsets.index) == True] = with_renamed_subsets

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=3000, n_bins=20, flavor='seurat',  inplace=True)

In [ ]:
sc.tl.pca(sub1, n_comps=55, svd_solver='arpack', random_state=42, use_highly_variable=True) #change use_highly_variable == True if you ran the block above

#### Harmony integrate the data if there are large batch effects

In [ ]:
#sce.pp.harmony_integrate(adata, 'sample') # Usually you want to run based on replicates but you can also run based on other parameters

If you batch corrected, make sure to use the use_rep = 'X_pca_harmony' paremeter below

In [ ]:
rng = np.random.RandomState(42)
sc.pp.neighbors(sub1, n_neighbors=30, random_state=42, use_rep='X_pca')  # Change use_rep == X_pca_harmony if you ran the block above

In [ ]:
sc.tl.umap(sub1, random_state=42)

In [ ]:
sc.set_figure_params(dpi=80, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    sub1, 
    color=['sample', 'cell_type', 'cell_type_subset', 'Foxp3'],   
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.5,
    outline_width=[0.6, 0.05],
    size=5,
    frameon=False,
    add_outline=True,
    sort_order = False
)

## Subset 2

In [ ]:
sub2 = adata[adata.obs['cell_type'].isin(['Subset2'])].copy()
sub2

In [ ]:
sc.pp.highly_variable_genes(sub2, n_top_genes=3000, n_bins=20, flavor='seurat',  inplace=True)

In [ ]:
sc.tl.pca(sub2, n_comps=55, svd_solver='arpack', random_state=42, use_highly_variable=True) #change use_highly_variable == True if you ran the block above

#### Harmony integrate the data if there are large batch effects

In [ ]:
#sce.pp.harmony_integrate(adata, 'condition') #Usually you want to run based on replicates but you can also run based on other parameters

If you batch corrected, make sure to use the use_rep = 'X_pca_harmony' paremeter below

In [ ]:
rng = np.random.RandomState(42)
sc.pp.neighbors(sub2, n_neighbors=30, random_state=42, use_rep='X_pca')  # Change use_rep == X_pca_harmony if you ran the block above

In [ ]:
sc.tl.umap(sub2, random_state=42)#, min_dist = 0.3, spread = 1.1)

In [ ]:
sc.set_figure_params(dpi=80, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    sub2, 
    color=['sample', 'tissue', 'condition', 'Cd4'],   
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.5,
    outline_width=[0.6, 0.05],
    size=5,
    frameon=False,
    add_outline=True,
    sort_order = False
)

In [ ]:
sc.set_figure_params(dpi=80, dpi_save=300, color_map='viridis', vector_friendly=True, transparent=True)
sc.pl.umap(
    sub2, 
    color=[
        'sample', 

        'Ptprc',

        #Early Lymphoid Progentors
        'Kit', 'Ly6a', 'Il7r', 'Rag1', 'Bcl11a', 'Cd44',

        #T cells: 
        'Cd3e', 'Cd4', 'Cd8a', 'Cd8b1', 'Foxp3', 'Nkg7', 'Gzmb', 'Gzmk', 

        #EM 
        'Il7r', 'Cd44', 'Sell', #Negative

        #Progenitor Exhuasted
        'Tcf7', 'Pdcd1', 'Havcr2',

        #Early T
        'Lef1', 'Sell', 'Tcf7',

        #Cd4 Th1
        'Tbx21', 'Ifng',

        #Cd4 Th2 
        'Gata3', 'Ifng',

        #Cd4 Th17
        'Rorc', 'Il17a',

        #Cd4 Th22
        'Ahr', 'Il22',

        ],  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=4,
    wspace = 0.3,
    outline_width=[0.6, 0.05],
    vmax='p99',
    frameon=False,
    add_outline=True,
    sort_order = False,
)

In [ ]:
#0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 
#1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0

for resolution_parameter in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5]:
    sc.tl.leiden(sub2, resolution=resolution_parameter, random_state=42, key_added='leiden_'+str(resolution_parameter))

In [ ]:
#'leiden_0.1', 'leiden_0.2', 'leiden_0.3','leiden_0.4', 'leiden_0.5', 'leiden_0.6', 'leiden_0.8', 'leiden_0.9', 'leiden_1.0',
#'leiden_1.1', 'leiden_1.2', 'leiden_1.3','leiden_1.4', 'leiden_1.5', 'leiden_1.6', 'leiden_1.7', 'leiden_1.8', 'leiden_1.9', 'leiden_2.0'

sc.set_figure_params(dpi=80, dpi_save=300, color_map='viridis', vector_friendly=True, transparent=True)
sc.pl.umap(
    sub2, 
    color=[
        'leiden_0.1', 'leiden_0.2', 'leiden_0.3', 'leiden_0.4', 'leiden_0.5',
        'leiden_0.6', 'leiden_0.7', 'leiden_0.8', 'leiden_0.9', 'leiden_1.0',
        'leiden_1.1', 'leiden_1.2', 'leiden_1.3','leiden_1.4', 'leiden_1.5'
           ], 
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=4,
    wspace = 0.7,
    outline_width=[0.6, 0.05],
    #size=35,
    frameon=False,
    add_outline=True,
    sort_order = False
)

In [ ]:
sc.tl.rank_genes_groups(sub1, 
                        groupby='leiden_1.0',  # Change the Leiden clustering
                        method='wilcoxon', 
                        use_raw=False)

result = sub1.uns['rank_genes_groups']
groups = result['names'].dtype.names

df = pd.DataFrame(
    {group + '_' + key[:1]: result[key][group]
    for group in groups for key in ['names']}).head(150)

#df.to_csv(file_outputs + '1_liver_leiden0.6.csv'', index = False)

df.head(25)

In [ ]:
#'leiden_0.1', 'leiden_0.2', 'leiden_0.3','leiden_0.4', 'leiden_0.5', 'leiden_0.6', 'leiden_0.8', 'leiden_0.9', 'leiden_1.0',
#'leiden_1.1', 'leiden_1.2', 'leiden_1.3','leiden_1.4', 'leiden_1.5', 'leiden_1.6', 'leiden_1.7', 'leiden_1.8', 'leiden_1.9', 'leiden_2.0'

sc.set_figure_params(dpi=80, dpi_save=300, color_map='viridis', vector_friendly=True, transparent=True)
sc.pl.umap(
    sub2, 
    color=[
        'leiden_1.1', 'Cd4', 'Cd8a', 'Sell', 'Lef1', 'Mki67', 'Foxp3',
           ], 
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=2,
    wspace = 0.7,
    outline_width=[0.6, 0.05],
    #size=35,
    frameon=False,
    add_outline=True,
    sort_order = False
)

In [ ]:
cell_type_groups = {
    'T CD4:CTL' : ['1', '5', '7', '9'], 
    'T CD4:Naive' : ['0', '2', '3', '4', '8', '11'],
    'T CD4:Ki67+': ['10'], 
    'T Treg': ['6']
    # add more as needed
}

cluster_to_cell_type = {cluster: cell_type for cell_type, clusters in cell_type_groups.items() for cluster in clusters}

In [ ]:
sub2.obs['cell_type'] = sub2.obs['leiden_1.1'].map(cluster_to_cell_type) #Add Leiden Cluster

In [ ]:
sc.set_figure_params(dpi=80, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    sub2, 
    color=[
        'cell_type', 
          ],
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.3,
    outline_width=[0.6, 0.05],
    frameon=False,
    add_outline=True,
    sort_order = False, 
)

## Subset 3 

In [ ]:
sub3 = adata[adata.obs['cell_type'].isin(['Subset3'])].copy()
sub3

In [ ]:
sc.pp.highly_variable_genes(sub3, n_top_genes=1000, n_bins=20, flavor='seurat',  inplace=True)

In [ ]:
sc.tl.pca(sub3, n_comps=25, svd_solver='arpack', random_state=42, use_highly_variable=True) #change use_highly_variable == True if you ran the block above

#### Harmony integrate the data if there are large batch effects

In [ ]:
#sce.pp.harmony_integrate(adata, 'condition') #Usually you want to run based on replicates but you can also run based on other parameters

If you batch corrected, make sure to use the use_rep = 'X_pca_harmony' paremeter below

In [ ]:
rng = np.random.RandomState(42)
sc.pp.neighbors(sub3, n_neighbors=30, random_state=42, use_rep='X_pca')  # Change use_rep == X_pca_harmony if you ran the block above

In [ ]:
sc.tl.umap(sub3, random_state=42)#, min_dist = 0.3, spread = 1.1)

In [ ]:
sc.set_figure_params(dpi=80, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    sub3, 
    color=['sample', 'tissue', 'condition', 'Cd4'],   
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.5,
    outline_width=[0.6, 0.05],
    size=10,
    frameon=False,
    add_outline=True,
    sort_order = False
)

In [ ]:
sc.set_figure_params(dpi=80, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    sub3, 
    color=[
            'Mki67',

            #Monocytes 
           'Cd14', 'Adgre1', #Negative 
           
           #cDCs
           'Sirpa', 'Clec9a', 'Xcr1', 'Itgam', 
           
           #pDCs
           'Siglech', 'Bst2', 'Tcf4', 'Irf8',

           #Basophils
           'Fcer1a',

           #Monocytes
           'Ly6c1', 'Cx3cr1', 'Cd209a', 'H2-Ab1', 'S100a8',
           
           #KC 
           'Clec4f', 'Timd4', 'Vsig4', 

           #ILC1
           'Tbx21', 'Ifng', 'Il1r1', 'Itga1',

           #Neutrophils
           'Ly6g', 'Cxcr2', 'Ceacam1'
           ],   
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.5,
    outline_width=[0.6, 0.05],
    size=10,
    frameon=False,
    add_outline=True,
    sort_order = False
)

In [ ]:
#0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 
#1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0

for resolution_parameter in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5]:
    sc.tl.leiden(sub3, resolution=resolution_parameter, random_state=42, key_added='leiden_'+str(resolution_parameter))

In [ ]:
#'leiden_0.1', 'leiden_0.2', 'leiden_0.3','leiden_0.4', 'leiden_0.5', 'leiden_0.6', 'leiden_0.8', 'leiden_0.9', 'leiden_1.0',
#'leiden_1.1', 'leiden_1.2', 'leiden_1.3','leiden_1.4', 'leiden_1.5', 'leiden_1.6', 'leiden_1.7', 'leiden_1.8', 'leiden_1.9', 'leiden_2.0'

sc.set_figure_params(dpi=80, dpi_save=300, color_map='viridis', vector_friendly=True, transparent=True)
sc.pl.umap(
    sub3, 
    color=[
        'leiden_0.1', 'leiden_0.2', 'leiden_0.3', 'leiden_0.4', 'leiden_0.5',
        'leiden_0.6', 'leiden_0.7', 'leiden_0.8', 'leiden_0.9', 'leiden_1.0',
        'leiden_1.1', 'leiden_1.2', 'leiden_1.3','leiden_1.4', 'leiden_1.5'
           ], 
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=4,
    wspace = 0.7,
    outline_width=[0.6, 0.05],
    #size=35,
    frameon=False,
    add_outline=True,
    sort_order = False
)

In [ ]:
sc.tl.rank_genes_groups(sub3, 
                        groupby='leiden_0.3',  # Change the Leiden clustering
                        method='wilcoxon', 
                        use_raw=False)

result = sub3.uns['rank_genes_groups']
groups = result['names'].dtype.names

df = pd.DataFrame(
    {group + '_' + key[:1]: result[key][group]
    for group in groups for key in ['names']}).head(150)

df.to_csv(file_outputs + '1_liver_sub3_leiden0.3.csv', index = False)

df.head(25)

In [ ]:
#'leiden_0.1', 'leiden_0.2', 'leiden_0.3','leiden_0.4', 'leiden_0.5', 'leiden_0.6', 'leiden_0.8', 'leiden_0.9', 'leiden_1.0',
#'leiden_1.1', 'leiden_1.2', 'leiden_1.3','leiden_1.4', 'leiden_1.5', 'leiden_1.6', 'leiden_1.7', 'leiden_1.8', 'leiden_1.9', 'leiden_2.0'

sc.set_figure_params(dpi=80, dpi_save=300, color_map='viridis', vector_friendly=True, transparent=True)
sc.pl.umap(
    sub3, 
    color=[
        'leiden_0.3', 'Cd4', 'Cd3e', 'Trac'
           ], 
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=2,
    wspace = 0.7,
    outline_width=[0.6, 0.05],
    #size=35,
    frameon=False,
    add_outline=True,
    sort_order = False
)

In [ ]:
cell_type_groups = {
    'Basophils' : ['4'], 
    'pDC' : ['2', '5'],
    'Monocytes': ['1'], 
    'ILC': ['3'], 
    'Myeloid:Ki67+': ['0']
    # add more as needed
}

cluster_to_cell_type = {cluster: cell_type for cell_type, clusters in cell_type_groups.items() for cluster in clusters}

In [ ]:
sub3.obs['cell_type'] = sub3.obs['leiden_0.3'].map(cluster_to_cell_type) #Add Leiden Cluster

In [ ]:
sc.set_figure_params(dpi=80, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    sub3, 
    color=[
        'cell_type', 
          ],
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.3,
    outline_width=[0.6, 0.05],
    frameon=False,
    add_outline=True,
    sort_order = False, 
)

# Map Subset 1, 2, and 3 to OG UMAP

In [ ]:
adata.obs['cell_type'].unique()

In [ ]:
temp = adata[~adata.obs['cell_type'].isin(['Subset1', 'Subset2', 'Subset3'])].obs['cell_type']
temp

In [ ]:
with_renamed_subsets = pd.concat([sub1.obs['cell_type_subset'],
                                  sub2.obs['cell_type'],
                                  sub3.obs['cell_type'],
                                  temp])

In [ ]:
adata.obs['cell_type_subset'] = ''

In [ ]:
adata.obs['cell_type_subset'][adata.obs.index.isin(with_renamed_subsets.index) == True] = with_renamed_subsets

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=3000, n_bins=20, flavor='seurat',  inplace=True)

In [ ]:
sc.tl.pca(adata, n_comps=45, svd_solver='arpack', random_state=42, use_highly_variable=True) #change use_highly_variable == True if you ran the block above

#### Harmony integrate the data if there are large batch effects

In [ ]:
#sce.pp.harmony_integrate(adata, 'condition') #Usually you want to run based on replicates but you can also run based on other parameters

If you batch corrected, make sure to use the use_rep = 'X_pca_harmony' paremeter below

In [ ]:
rng = np.random.RandomState(42)
sc.pp.neighbors(adata, n_neighbors=30, random_state=42, use_rep='X_pca')  # Change use_rep == X_pca_harmony if you ran the block above

In [ ]:
sc.tl.umap(adata, random_state=42)#, min_dist = 0.3, spread = 1.1)

In [ ]:
sc.set_figure_params(dpi=80, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    adata, 
    color=['sample', 'cell_type_subset', 'Cd4', 'Cd8a', 'Foxp3'],   
    color_map='Spectral_r', 
    use_raw=False,
    ncols=3,
    wspace = 0.5,
    outline_width=[0.6, 0.05],
    size=5,
    frameon=False,
    add_outline=True,
    sort_order = False
)

# Clean Adata

In [ ]:
adata

In [ ]:
adata.obs = adata.obs[['condition', 'tissue', 'sample', 'cell_type', 'cell_type_subset']]
adata

In [ ]:
adata.uns = {k: adata.uns[k] for k in ['condition_colors', 'tissue_colors', 'sample_colors', 'cell_type_colors', 'cell_type_subset_colors',
                                       'log1p', 'neighbors', 'pca', 'umap'] if k in adata.uns}

adata

In [ ]:
adata.write_h5ad(h5ad + '3_liver.h5ad')